In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path("../")

WAREHOUSE_DIR = PROJECT_ROOT / "data" / "warehouse"

In [2]:
fact_orders = pd.read_csv(
    WAREHOUSE_DIR / "fact_orders.csv", parse_dates=["order_purchase_timestamp"]
)


dim_customer = pd.read_csv(WAREHOUSE_DIR / "dim_customer.csv")


dim_product = pd.read_csv(WAREHOUSE_DIR / "dim_product.csv")


dim_seller = pd.read_csv(WAREHOUSE_DIR / "dim_seller.csv")

In [3]:
fact_orders.head()

,order_id,customer_unique_id,product_id,seller_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,price,freight_value,order_revenue,delivery_days,review_score,payment_type,payment_installments,payment_value,payment_records
0,e481f51cbdc54678b7cc49136f2d6af7,7c396fd4830fd04220f754e42b4e5bff,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,29.99,8.72,38.71,8.0,4.0,credit_card,1.0,38.71,3.0
1,53cdb2fc8bc7dce0b6741e2150273451,af07308b275d755c9edb36a90c618231,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,118.70,22.76,141.46,13.0,4.0,boleto,1.0,141.46,1.0
2,47770eb9100c2d0c44946d9cf07ec65d,3a653a41f6f9fc3d2a113cf8398680e8,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,159.90,19.22,179.12,9.0,5.0,credit_card,3.0,179.12,1.0
3,949d5b44dbf5de918fe9c16f97b45f8a,7c142cf63193a1473d2e66489a9ae977,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,45.00,27.20,72.20,13.0,5.0,credit_card,1.0,72.20,1.0
4,ad21c59c0840e6cb83a9ceb5573f8159,72632f0f9dd73dfee390c9b22eb56dd6,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,19.90,8.72,28.62,2.0,5.0,credit_card,1.0,28.62,1.0


### Define Analysis Reference Date
- This is the cutoff date (latest transaction date) to know the last purchase's date

In [4]:
reference_date = fact_orders["order_purchase_timestamp"].max()


reference_date

Timestamp('2018-10-17 17:30:18')

### Customer Base

In [5]:
customers_ml = (
    fact_orders["customer_unique_id"].drop_duplicates().reset_index(drop=True)
)

customers_ml = pd.DataFrame({"customer_unique_id": customers_ml})

### RFM Features 
- Recency = How recently did the customer buy?
- Frequency = Number of orders
- Monetary = Total spending

In [6]:
recency = (
    fact_orders.groupby("customer_unique_id")["order_purchase_timestamp"]
    .max()
    .reset_index()
)

recency["recency_days"] = (reference_date - recency["order_purchase_timestamp"]).dt.days

recency = recency[["customer_unique_id", "recency_days"]]

In [7]:
frequency = (
    fact_orders.groupby("customer_unique_id")["order_id"]
    .nunique()
    .reset_index()
    .rename(columns={"order_id": "frequency"})
)

In [8]:
monetary = (
    fact_orders.groupby("customer_unique_id")["order_revenue"]
    .sum()
    .reset_index()
    .rename(columns={"order_revenue": "monetary"})
)

In [9]:
# Merge them
customers_ml = (
    customers_ml.merge(recency, on="customer_unique_id", how="left")
    .merge(frequency, on="customer_unique_id", how="left")
    .merge(monetary, on="customer_unique_id", how="left")
)

In [10]:
customers_ml.head()

,customer_unique_id,recency_days,frequency,monetary
0,7c396fd4830fd04220f754e42b4e5bff,380,2,82.82
1,af07308b275d755c9edb36a90c618231,84,1,141.46
2,3a653a41f6f9fc3d2a113cf8398680e8,70,1,179.12
3,7c142cf63193a1473d2e66489a9ae977,332,1,72.20
4,72632f0f9dd73dfee390c9b22eb56dd6,245,1,28.62


### Average Order Value 

In [11]:
avg_order_value = (
    fact_orders.groupby("customer_unique_id")["order_revenue"]
    .agg(["mean", "count"])
    .reset_index()
)

In [12]:
avg_order_value.columns = ["customer_unique_id", "avg_order_value", "total_items"]

In [13]:
customers_ml = customers_ml.merge(avg_order_value, on="customer_unique_id", how="left")

### Product Behaviour Features

In [14]:
product_diversity = (
    fact_orders.groupby("customer_unique_id")["product_id"]
    .nunique()
    .reset_index()
    .rename(columns={"product_id": "unique_products"})
)

product_diversity = (
    fact_orders.groupby("customer_unique_id")["product_id"]
    .nunique()
    .reset_index()
    .rename(columns={"product_id": "unique_products"})
)

In [15]:
product_behavior = fact_orders.merge(
    dim_product[["product_key", "product_category_name_english"]],
    left_on="product_id",
    right_on="product_key",
    how="left",
)

category_diversity = (
    product_behavior.groupby("customer_unique_id")["product_category_name_english"]
    .nunique()
    .reset_index()
    .rename(columns={"product_category_name_english": "unique_categories"})
)


customers_ml = customers_ml.merge(
    category_diversity, on="customer_unique_id", how="left"
)

### Seller Behaviour Feature
- How many sellers did the customer purchase from

In [16]:
seller_diversity = (
    fact_orders.groupby("customer_unique_id")["seller_id"]
    .nunique()
    .reset_index()
    .rename(columns={"seller_id": "unique_sellers"})
)

customers_ml = customers_ml.merge(seller_diversity, on="customer_unique_id", how="left")

### Customer Experience Features

#### Average Review Score

In [17]:
review_features = (
    fact_orders.groupby("customer_unique_id")["review_score"]
    .mean()
    .reset_index()
    .rename(columns={"review_score": "avg_review_score"})
)

In [18]:
customers_ml = customers_ml.merge(review_features, on="customer_unique_id", how="left")

#### Delivery Problems - Late Delivery Ratio

In [19]:
fact_orders["is_late"] = pd.to_datetime(
    fact_orders["order_delivered_customer_date"]
) > pd.to_datetime(fact_orders["order_estimated_delivery_date"])

In [20]:
late_ratio = (
    fact_orders.groupby("customer_unique_id")["is_late"]
    .mean()
    .reset_index()
    .rename(columns={"is_late": "late_delivery_ratio"})
)

In [21]:
customers_ml = customers_ml.merge(late_ratio, on="customer_unique_id", how="left")

### Payment Features

- total_payment_value
- avg_payment_value
- avg_installments
- max_installments
- payment_method_count
- preferred_payment_type

In [22]:
payment_features = (
    fact_orders
    .groupby("customer_unique_id")
    .agg(
        total_payment_value=("payment_value", "sum"),
        avg_payment_value=("payment_value", "mean"),
        avg_installments=("payment_installments", "mean"),
        max_installments=("payment_installments", "max"),
        payment_method_count=("payment_type", "nunique")
    )
    .reset_index()
)

In [23]:
preferred_payment = (
    fact_orders
    .groupby("customer_unique_id")["payment_type"]
    .agg(lambda x: x.mode()[0])
    .reset_index()
    .rename(
        columns={
            "payment_type": "preferred_payment_type"
        }
    )
)

In [24]:
customers_ml = customers_ml.merge(
    payment_features,
    on="customer_unique_id",
    how="left"
)

customers_ml = customers_ml.merge(
    preferred_payment,
    on="customer_unique_id",
    how="left"
)

### Geography Features

#### Customer Location

In [25]:
dim_customer_ml = (
    dim_customer.groupby("customer_unique_id")
    .agg(
        {
            "city":"first",
            "state":"first",
            "latitude":"first",
            "longitude":"first"
        }
    )
    .reset_index()
)

In [26]:
customer_geo = dim_customer_ml[["customer_unique_id", "state", "latitude", "longitude"]]

In [27]:
customers_ml = customers_ml.merge(customer_geo, on="customer_unique_id", how="left")

### Create Data-Driven Churn Target

In [30]:
# ============================================================
# STEP 7 — Create Data-Driven Churn Target
# ============================================================
#
# Objective:
# Create a customer-level churn_label for supervised learning.
#
# Definition:
# A customer is considered "churned" if their most recent purchase
# occurred more than the data-driven inactivity threshold before
# the end of the observation period.
#
# The threshold is derived from positive repurchase intervals among
# customers who made more than one purchase.
#
# churn_label:
#   1 = Churned / inactive beyond the threshold
#   0 = Not churned / still within the expected repurchase window
#
# Important:
# Olist does not contain a ground-truth churn indicator.
# Therefore, this is an inferred inactivity-based target.
# ============================================================

# ------------------------------------------------------------
# 1. Prepare unique customer-order purchase history
# ------------------------------------------------------------

orders_for_churn = fact_orders[
    ["order_id", "customer_unique_id", "order_purchase_timestamp"]
].drop_duplicates()

# Convert timestamp to datetime
orders_for_churn["order_purchase_timestamp"] = pd.to_datetime(
    orders_for_churn["order_purchase_timestamp"],
    errors="coerce"
)

# Remove records with missing information required for churn calculation
orders_for_churn = orders_for_churn.dropna(
    subset=[
        "order_id",
        "customer_unique_id",
        "order_purchase_timestamp"
    ]
)

# ------------------------------------------------------------
# 2. Calculate purchase intervals
# ------------------------------------------------------------

orders_for_churn = orders_for_churn.sort_values(
    ["customer_unique_id", "order_purchase_timestamp"]
).copy()

# Previous purchase date for each customer
orders_for_churn["previous_purchase_date"] = orders_for_churn.groupby(
    "customer_unique_id"
)["order_purchase_timestamp"].shift(1)

# Number of days between consecutive purchases
orders_for_churn["days_between_purchases"] = (
    orders_for_churn["order_purchase_timestamp"]
    - orders_for_churn["previous_purchase_date"]
).dt.days

# Keep only actual repeat-purchase intervals.
# Zero-day intervals represent multiple orders on the same day
# and are not useful for estimating customer inactivity.
positive_intervals = orders_for_churn.loc[
    orders_for_churn["days_between_purchases"] > 0,
    "days_between_purchases"
]

# ------------------------------------------------------------
# 3. Determine the data-driven churn threshold
# ------------------------------------------------------------

# Use the 90th percentile of positive repurchase intervals.
#
# Interpretation:
# Approximately 80% of observed positive repurchase intervals
# among repeat customers are shorter than this threshold.
#
# This is an inactivity threshold rather than a directly observed
# churn event.
churn_threshold = positive_intervals.quantile(0.80)

# Round to a whole number of days
churn_threshold = int(round(churn_threshold))

print(f"Data-driven churn threshold: {churn_threshold} days")

# ------------------------------------------------------------
# 4. Determine the observation period
# ------------------------------------------------------------

# Use the latest purchase date available in the dataset as the
# end of the observation period.
observation_end_date = orders_for_churn[
    "order_purchase_timestamp"
].max()

print(f"Observation end date: {observation_end_date.date()}")

# ------------------------------------------------------------
# 5. Calculate each customer's recency
# ------------------------------------------------------------

customer_last_purchase = (
    orders_for_churn
    .groupby("customer_unique_id")["order_purchase_timestamp"]
    .max()
    .reset_index()
)

customer_last_purchase = customer_last_purchase.rename(
    columns={
        "order_purchase_timestamp": "last_purchase_date"
    }
)

# Number of days since each customer's most recent purchase
customer_last_purchase["recency_days"] = (
    observation_end_date
    - customer_last_purchase["last_purchase_date"]
).dt.days

# ------------------------------------------------------------
# 6. Create the churn target
# ------------------------------------------------------------

customer_last_purchase["churn_label"] = (
    customer_last_purchase["recency_days"] > churn_threshold
).astype(int)

# ------------------------------------------------------------
# 7. Merge the target into customers_ml
# ------------------------------------------------------------

# Remove an existing churn_label if one already exists
# so that the target is not duplicated during the merge.
customers_ml = customers_ml.drop(
    columns=["churn_label", "recency_days", "last_purchase_date"],
    errors="ignore"
)

customers_ml = customers_ml.merge(
    customer_last_purchase[
        [
            "customer_unique_id",
            "last_purchase_date",
            "recency_days",
            "churn_label"
        ]
    ],
    on="customer_unique_id",
    how="left"
)

# ------------------------------------------------------------
# 8. Validate the target
# ------------------------------------------------------------

print("\nChurn Label Distribution:")
print(customers_ml["churn_label"].value_counts())

print("\nChurn Label Distribution (%):")
print(
    customers_ml["churn_label"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)


Data-driven churn threshold: 197 days
Observation end date: 2018-10-17

Churn Label Distribution:
churn_label
1    64212
0    31884
Name: count, dtype: int64

Churn Label Distribution (%):
churn_label
1    66.82
0    33.18
Name: proportion, dtype: float64


### Missing Values Handling

In [32]:
customers_ml.isnull().sum()

customer_unique_id          0
frequency                   0
monetary                    0
avg_order_value           676
total_items                 0
unique_categories           0
unique_sellers              0
avg_review_score          716
late_delivery_ratio         0
total_payment_value         0
avg_payment_value           0
avg_installments            0
max_installments            0
payment_method_count        0
preferred_payment_type      0
state                     268
latitude                  268
longitude                 268
last_purchase_date          0
recency_days                0
churn_label                 0
dtype: int64

In [33]:
numeric_columns = customers_ml.select_dtypes(include=np.number).columns


customers_ml[numeric_columns] = customers_ml[numeric_columns].fillna(0)

In [34]:
categorical_columns = customers_ml.select_dtypes(exclude=np.number).columns


customers_ml[categorical_columns] = customers_ml[categorical_columns].fillna("unknown")

### ML Feature Dataset Review

In [35]:
customers_ml.shape

(96096, 21)

In [40]:
customers_ml

,customer_unique_id,frequency,monetary,avg_order_value,total_items,unique_categories,unique_sellers,avg_review_score,late_delivery_ratio,total_payment_value,...,avg_installments,max_installments,payment_method_count,preferred_payment_type,state,latitude,longitude,last_purchase_date,recency_days,churn_label
0,7c396fd4830fd04220f754e42b4e5bff,2,82.82,41.41,2,2,2,4.5,0.0,82.82,...,1.0,1.0,1,credit_card,SP,-23.577482,-46.587077,2017-10-02 10:56:33,380,1
1,af07308b275d755c9edb36a90c618231,1,141.46,141.46,1,1,1,4.0,0.0,141.46,...,1.0,1.0,1,boleto,BA,-12.186877,-44.540232,2018-07-24 20:41:37,84,0
2,3a653a41f6f9fc3d2a113cf8398680e8,1,179.12,179.12,1,1,1,5.0,0.0,179.12,...,3.0,3.0,1,credit_card,GO,-16.745150,-48.514783,2018-08-08 08:38:49,70,0
3,7c142cf63193a1473d2e66489a9ae977,1,72.20,72.20,1,1,1,5.0,0.0,72.20,...,1.0,1.0,1,credit_card,RN,-5.774002,-35.270976,2017-11-18 19:28:06,332,1
4,72632f0f9dd73dfee390c9b22eb56dd6,1,28.62,28.62,1,1,1,5.0,0.0,28.62,...,1.0,1.0,1,credit_card,SP,-23.676257,-46.514580,2018-02-13 21:18:39,245,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96091,6359f309b166b0196dbf7ad2ac62bb5a,1,85.08,85.08,1,1,1,5.0,0.0,85.08,...,3.0,3.0,1,credit_card,SP,-23.177943,-45.882139,2017-03-09 09:54:05,587,1
96092,da62f9e57a76d978d02ab5362c509660,1,195.00,195.00,1,1,1,4.0,0.0,195.00,...,3.0,3.0,1,credit_card,SP,-24.001334,-46.450022,2018-02-06 12:58:58,253,1
96093,737520a9aad80b3fbbdad19b66b37b30,1,271.01,271.01,1,1,1,5.0,0.0,271.01,...,5.0,5.0,1,credit_card,BA,-17.898045,-39.373106,2017-08-27 14:46:43,416,1
96094,5097a5312c8b157bb7be58ae360ef43c,1,441.16,220.58,2,1,1,2.0,0.0,882.32,...,4.0,4.0,1,credit_card,RJ,-22.563909,-42.695343,2018-01-08 21:28:27,281,1


In [37]:
customers_ml["churn_label"].value_counts(normalize=True)

churn_label
1    0.668207
0    0.331793
Name: proportion, dtype: float64

In [38]:
FEATURE_DIR = PROJECT_ROOT / "data" / "features"


FEATURE_DIR.mkdir(parents=True, exist_ok=True)

customers_ml.to_csv(FEATURE_DIR / "customer_ml_features.csv", index=False)